# FPL data split and benchmarks

The first forecasting task estimates points for a recorded player-fixture before its gameweek.
Season partitions are fixed before features or benchmarks are calculated. Historical targets
use `P_adj`; current targets use official `P_off`, following the
[target policy](../docs/fpl_rules.md#adjusted-target-used-in-notebook-1).

Validation covers 2024/25. The 2025/26 test season and captured 2026/27 records remain unscored.
The average FPL manager score belongs to a later squad-level comparison, after captaincy,
substitutions, chips and transfer deductions are represented.

## Data import

Notebook 2's player-fixture records and fixture calendar supply the inputs. Hashes identify
the input versions and confirm that this stage preserves them.

In [1]:
from pathlib import Path
from hashlib import sha256
import json
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".cache/matplotlib"))

import numpy as np
import pandas as pd
from IPython.display import display

PROCESSED_DIR = PROJECT_ROOT / "data/processed"
MODEL_DIR = PROCESSED_DIR / "modelling"
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 30)
pd.set_option("display.float_format", "{:.4f}".format)

In [2]:
from src.data_preparation import MATCH_KEY, PLAYER_KEY, require_keys
from src.model_preparation import (
    FEATURES, PREDICTIONS, RULE_VERSION, SEASON_SPLITS, fit_benchmarks,
    historical_features, modelling_rows, predict_benchmarks,
    prediction_calendar, split_checks,
)

input_paths = [PROCESSED_DIR / name for name in
               ["fpl_eda_tables.pkl", "fpl_eda_manifest.json"]]
source_hashes = {path.name: sha256(path.read_bytes()).hexdigest() for path in input_paths}
tables = pd.read_pickle(input_paths[0])
eda_manifest = json.loads(input_paths[1].read_text())
display(pd.DataFrame([
    {"table": name, "rows": len(tables[name])}
    for name in ["player_matches", "fixtures", "current_events"]
]))

,table,rows
0,player_matches,119822
1,fixtures,1900
2,current_events,38


## Fix chronological partitions

Training fits the references. Validation compares later forecasts and supports subsequent
model development. The test season is reserved until those choices are fixed; monitoring
records belong to a separate later season. No rows are shuffled.

The same footballer may occur in different seasons. This matches forecasting future seasons
for existing players. Season-scoped IDs remain join keys, and initial histories reset each
season. This is not a test of generalisation to entirely unknown players.

In [3]:
SPLIT_ORDER = ["train", "validation", "test", "monitor"]
RESULT_DELAY_HOURS = 24
partition_plan = pd.Series(SEASON_SPLITS, name="split").rename_axis("season").reset_index()
display(partition_plan)

,season,split
0,2022_2023,train
1,2023_2024,train
2,2024_2025,validation
3,2025_2026,test
4,2026_2027,monitor


## Establish the information cutoff

Each gameweek has one forecast cutoff, shared by all its fixtures. Saved FPL event deadlines
are available for 2026/27. Historical deadlines are absent, so their labelled retrospective
proxy is the first recorded kickoff minus 24 hours.

A result enters history only after kickoff plus a fixed 24-hour delay. This excludes near-cutoff
matches and all results within the target gameweek. The delay is an availability assumption;
archived corrections and final fixture assignments do not establish exact historical
publication times. The checks below validate this stated protocol rather than certify an
exact historical replay.

In [4]:
calendar = prediction_calendar(tables["fixtures"], tables["current_events"])
rows = modelling_rows(tables["player_matches"], tables["fixtures"], calendar)
require_keys(rows, MATCH_KEY, "partitioned player fixtures")
assert len(rows) == len(tables["player_matches"])
cutoff_coverage = calendar.groupby(["season", "cutoff_basis"]).size().rename("gameweeks")
display(cutoff_coverage.to_frame())
display(calendar.loc[calendar.GW.eq(1), [
    "season", "GW", "prediction_cutoff", "first_kickoff", "cutoff_basis",
]].reset_index(drop=True))

,,gameweeks
season,cutoff_basis,
2022_2023,first_kickoff_minus_24h_proxy,37
2023_2024,first_kickoff_minus_24h_proxy,38
2024_2025,first_kickoff_minus_24h_proxy,38
2025_2026,first_kickoff_minus_24h_proxy,38
2026_2027,saved_fpl_deadline,10


,season,GW,prediction_cutoff,first_kickoff,cutoff_basis
0,2022_2023,1,2022-08-04 19:00:00+00:00,2022-08-05 19:00:00+00:00,first_kickoff_minus_24h_proxy
1,2023_2024,1,2023-08-10 19:00:00+00:00,2023-08-11 19:00:00+00:00,first_kickoff_minus_24h_proxy
2,2024_2025,1,2024-08-15 19:00:00+00:00,2024-08-16 19:00:00+00:00,first_kickoff_minus_24h_proxy
3,2025_2026,1,2025-08-14 19:00:00+00:00,2025-08-15 19:00:00+00:00,first_kickoff_minus_24h_proxy
4,2026_2027,1,2026-08-21 17:30:00+00:00,2026-08-21 19:00:00+00:00,saved_fpl_deadline


## Inspect split boundaries

The table shows record counts and actual time boundaries. Complete fixtures and gameweeks
stay together, including doubles. The 2022/23 cancelled gameweek has no observed fixtures;
missing records are not added as zero-point targets.

In [5]:
split_summary = rows.groupby("split").agg(
    records=("points", "size"), seasons=("season", "nunique"),
    first_cutoff=("prediction_cutoff", "min"), last_cutoff=("prediction_cutoff", "max"),
    first_target=("kickoff_time", "min"), last_target=("kickoff_time", "max"),
).reindex(SPLIT_ORDER)
display(split_summary)
split_validation = split_checks(rows, RESULT_DELAY_HOURS)
display(split_validation)
assert split_validation.passed.all()

,records,seasons,first_cutoff,last_cutoff,first_target,last_target
split,,,,,,
train,56230,2,2022-08-04 19:00:00+00:00,2024-05-18 15:00:00+00:00,2022-08-05 19:00:00+00:00,2024-05-19 15:00:00+00:00
validation,27283,1,2024-08-15 19:00:00+00:00,2025-05-24 15:00:00+00:00,2024-08-16 19:00:00+00:00,2025-05-25 15:00:00+00:00
test,29747,1,2025-08-14 19:00:00+00:00,2026-05-23 15:00:00+00:00,2025-08-15 19:00:00+00:00,2026-05-24 15:00:00+00:00
monitor,6562,1,2026-08-21 17:30:00+00:00,2026-11-06 18:30:00+00:00,2026-08-21 19:00:00+00:00,2026-11-08 16:30:00+00:00


,check,passed
0,configured_season_assignment,True
1,one_split_per_fixture,True
2,one_split_per_gameweek,True
3,one_cutoff_per_gameweek,True
4,cutoff_precedes_target,True
5,finite_targets,True
6,train_results_before_validation,True
7,validation_results_before_test,True
8,test_results_before_monitor,True


## Build features from available history

The initial features are recent points, recent minutes and prior record count. Recent means
use up to five observed same-season fixtures, including non-appearances. An as-of join selects
the latest history strictly before each cutoff; it is not a one-row shift that could expose
the first result of a double gameweek.

Current snapshots, target-fixture statistics, price, supplied form/xP and full-season profiles
are outside the feature set. Future models will receive only the explicit feature columns.

In [6]:
features = historical_features(rows, result_delay_hours=RESULT_DELAY_HOURS)
has_history = features.history_available_at.notna()
same_week = features.groupby(PLAYER_KEY + ["GW"])[FEATURES].nunique(dropna=False)
feature_validation = pd.DataFrame([
    {"check": "history_strictly_before_cutoff", "passed": features.loc[
        has_history, "history_available_at"
    ].lt(features.loc[has_history, "prediction_cutoff"]).all()},
    {"check": "target_fixture_excluded", "passed": ~features.loc[
        has_history, "history_fixture_id"
    ].eq(features.loc[has_history, "fpl_fixture_id"]).any()},
    {"check": "same_gameweek_features_frozen", "passed": same_week.le(1).all().all()},
    {"check": "all_source_rows_preserved", "passed": len(features) == len(rows)},
])
display(feature_validation)
assert feature_validation.passed.all()

,check,passed
0,history_strictly_before_cutoff,True
1,target_fixture_excluded,True
2,same_gameweek_features_frozen,True
3,all_source_rows_preserved,True


## Inspect the evidence for individual forecasts

The historical fixture and its assumed availability time identify the newest result used
for a prediction. Missing history remains visible. The training-only position average will
provide its benchmark fallback.

In [7]:
trace_columns = ["name", "GW", "fpl_fixture_id", "prediction_cutoff",
                 "history_fixture_id", "history_available_at", *FEATURES]
validation_mask = features.split.eq("validation")
trace = pd.concat([
    features.loc[validation_mask & ~has_history].head(2),
    features.loc[validation_mask & has_history].head(4),
])
display(trace[trace_columns].reset_index(drop=True))
display(features.groupby("split").agg(
    records=("points", "size"),
    without_history=("history_available_at", lambda values: values.isna().sum()),
).reindex(SPLIT_ORDER))

,name,GW,fpl_fixture_id,prediction_cutoff,history_fixture_id,history_available_at,recent_points_5,recent_minutes_5,prior_records
0,Fábio Ferreira Vieira,1,2,2024-08-15 19:00:00+00:00,<NA>,NaT,NaN,NaN,0
1,Gabriel Fernando de Jesus,1,2,2024-08-15 19:00:00+00:00,<NA>,NaT,NaN,NaN,0
2,Fábio Ferreira Vieira,2,11,2024-08-23 11:30:00+00:00,2,2024-08-18 14:00:00+00:00,0.0000,0.0000,1
3,Gabriel Fernando de Jesus,2,11,2024-08-23 11:30:00+00:00,2,2024-08-18 14:00:00+00:00,0.0000,5.0000,1
4,Gabriel dos Santos Magalhães,2,11,2024-08-23 11:30:00+00:00,2,2024-08-18 14:00:00+00:00,6.0000,90.0000,1
5,Kai Havertz,2,11,2024-08-23 11:30:00+00:00,2,2024-08-18 14:00:00+00:00,12.0000,90.0000,1


,records,without_history
split,,
train,56230,1678
validation,27283,791
test,29747,844
monitor,6562,672


## Fit benchmark references on training records

Zero is a sparse-target reference. The global and positional means use training targets only.
The main benchmark averages the player's latest available five records, falling back to the
training position mean when history is absent.

During validation, earlier validation results may enter later histories after the availability
delay. The fitted training means remain fixed. This represents rolling weekly forecasting.

In [8]:
train = features.loc[features.split.eq("train")].copy()
validation = features.loc[validation_mask].copy()
fitted = fit_benchmarks(train)
validation_predictions = predict_benchmarks(validation, fitted)
display(pd.Series(fitted["position_mean"], name="training_mean_points").to_frame())
display(validation_predictions[["name", "GW", "points", *PREDICTIONS]].head(8))
assert validation_predictions.split.eq("validation").all()
assert validation_predictions[PREDICTIONS].notna().all().all()

,training_mean_points
DEF,1.1358
FWD,1.2918
GK,0.8472
MID,1.3417


,name,GW,points,zero,train_mean,train_position_mean,recent_five
56230,Fábio Ferreira Vieira,1,0.0000,0.0000,1.2121,1.3417,1.3417
56231,Gabriel Fernando de Jesus,1,0.0000,0.0000,1.2121,1.2918,1.2918
56232,Gabriel dos Santos Magalhães,1,6.0000,0.0000,1.2121,1.1358,1.1358
56233,Kai Havertz,1,12.0000,0.0000,1.2121,1.2918,1.2918
56234,Karl Hein,1,0.0000,0.0000,1.2121,0.8472,0.8472
56235,Jurriën Timber,1,1.0000,0.0000,1.2121,1.1358,1.1358
56236,Jorge Luiz Frello Filho,1,0.0000,0.0000,1.2121,1.3417,1.3417
56237,Jakub Kiwior,1,0.0000,0.0000,1.2121,1.1358,1.1358


## Challenge the temporal boundary

Adding 1,000 points to every test and monitoring target must leave training/validation features,
fitted means and validation predictions identical. This checks that later targets cannot flow
backwards into development. Same-cutoff and double-gameweek edge cases are also covered by
the local validation tests.

In [9]:
changed = rows.copy()
changed.loc[changed.split.isin(["test", "monitor"]), "points"] += 1000
challenged = historical_features(changed, result_delay_hours=RESULT_DELAY_HOURS)
development = features.split.isin(["train", "validation"])
pd.testing.assert_frame_equal(
    features.loc[development, MATCH_KEY + FEATURES],
    challenged.loc[development, MATCH_KEY + FEATURES],
)
assert fitted == fit_benchmarks(challenged.loc[challenged.split.eq("train")])
challenged_predictions = predict_benchmarks(challenged.loc[validation_mask], fitted)
pd.testing.assert_frame_equal(
    validation_predictions[PREDICTIONS], challenged_predictions[PREDICTIONS],
)
display(pd.DataFrame([{"check": "future_target_perturbation", "passed": True}]))

,check,passed
0,future_target_perturbation,True


## Save modelling inputs

The modelling export separates development, reserved test and monitoring rows. Only validation
predictions are generated. Input hashes and the timing policy accompany the saved artifacts.

In [10]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_data = {
    "development": features.loc[development].copy(),
    "reserved_test": features.loc[features.split.eq("test")].copy(),
    "monitor": features.loc[features.split.eq("monitor")].copy(), "calendar": calendar,
}
model_audits = {"split": split_validation, "features": feature_validation,
                "split_summary": split_summary, "benchmark_parameters": fitted}
pd.to_pickle(model_data, MODEL_DIR / "fpl_model_data.pkl")
pd.to_pickle(validation_predictions, MODEL_DIR / "fpl_benchmark_predictions.pkl")
pd.to_pickle(model_audits, MODEL_DIR / "fpl_split_audits.pkl")

In [11]:
artifact_names = ["fpl_model_data.pkl", "fpl_benchmark_predictions.pkl", "fpl_split_audits.pkl"]
manifest = {
    "schema_version": RULE_VERSION, "created_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "input_sha256": source_hashes, "eda_schema": eda_manifest["schema_version"],
    "code_sha256": sha256((PROJECT_ROOT / "src/model_preparation.py").read_bytes()).hexdigest(),
    "split_seasons": SEASON_SPLITS, "features": FEATURES, "target": "points",
    "target_policy": "historical P_adj; current P_off",
    "historical_cutoff": "first recorded gameweek kickoff minus 24h; proxy",
    "current_cutoff": "saved FPL event deadline", "result_delay_hours": RESULT_DELAY_HOURS,
    "evaluation": "rolling gameweek forecasts; frozen training references",
    "test_scored": False, "manager_average": "deferred to squad-level evaluation",
    "artifacts_sha256": {name: sha256((MODEL_DIR / name).read_bytes()).hexdigest()
                         for name in artifact_names},
}
(MODEL_DIR / "fpl_model_manifest.json").write_text(json.dumps(manifest, indent=2))
assert source_hashes == {path.name: sha256(path.read_bytes()).hexdigest() for path in input_paths}
display(pd.DataFrame([{"partition": key, "records": len(frame)}
                      for key, frame in model_data.items() if key != "calendar"]))

,partition,records
0,development,83513
1,reserved_test,29747
2,monitor,6562


## Preparation summary

All source records retain their season assignment. Split checks, cutoff traces, frozen
double-gameweek histories and the future-target challenge pass under the stated timing policy.
Notebook 4 compares validation errors; the reserved test and monitoring partitions stay unscored.

Historical deadline and publication-time gaps limit exact replay. The test season has already
appeared in EDA and is reserved for predictive development decisions, rather than being wholly
unseen research data. Later collection of dated snapshots supports stronger prospective checks.